# Ticketmaster Ingestion Testing

**Source**: Ticketmaster Discovery API  
**Type**: REST API — requires `TICKETMASTER_API_KEY`  
**Description**: Ticketmaster is one of the largest global ticketing platforms covering concerts, sports, theatre, and more. This notebook tests the Ticketmaster ingestion pipeline end-to-end.

## What this notebook does
1. Initialises the environment and verifies the `TICKETMASTER_API_KEY` env var
2. Creates the Ticketmaster pipeline via `PipelineFactory`
3. Executes the pipeline with a limited scope (max 10 events, 1 page)
4. Inspects normalised `EventSchema` results
5. Explores `custom_fields` specific to Ticketmaster
6. Saves the batch to `data/batches/ticketmaster/`
7. Produces a DataFrame summary

In [1]:
import sys
import os
import logging
from pathlib import Path

# Add services/api to path so src.* imports resolve
REPO_ROOT = Path("__file__").resolve().parents[1]
API_ROOT = REPO_ROOT / "services" / "api"
if str(API_ROOT) not in sys.path:
    sys.path.insert(0, str(API_ROOT))

# Load .env from services/api
try:
    from dotenv import load_dotenv
    load_dotenv(API_ROOT / ".env")
    print("dotenv loaded")
except ImportError:
    print("python-dotenv not installed — skipping .env load")

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("notebook.ticketmaster")

# Check required API key
api_key = os.environ.get("TICKETMASTER_API_KEY")
if api_key:
    print(f"TICKETMASTER_API_KEY: set (length={len(api_key)})")
else:
    print("WARNING: TICKETMASTER_API_KEY is not set — pipeline execution will fail.")
    print("Set it in services/api/.env or export it in your shell.")
print(f"API_ROOT: {API_ROOT}")

dotenv loaded
TICKETMASTER_API_KEY: set (length=32)
API_ROOT: /Users/josegarcia/Documents/GitHub/event-intelligence-platform/services/api


In [2]:
from src.ingestion.factory import PipelineFactory

factory = PipelineFactory()

# Temporarily enable the source for testing (disabled by default in YAML)
source_cfg = factory.get_source_config("ticketmaster")
source_cfg["enabled"] = True

pipeline = factory.create_pipeline("ticketmaster")
print(f"Pipeline created: {pipeline.__class__.__name__}")
print(f"Source type: {pipeline.source_type}")
print(f"Source name: {pipeline.config.source_name}")

Pipeline created: BaseAPIPipeline
Source type: SourceType.API
Source name: ticketmaster


In [3]:
import asyncio

# Execute: 120 days ahead, Barcelona + Madrid, full pages (page_size capped at 200 by config)
result = await pipeline.execute()

print(f"Status        : {result.status}")
print(f"Execution ID  : {result.execution_id}")
print(f"Total fetched : {result.total_events_processed}")
print(f"Successful    : {result.successful_events}")
print(f"Failed        : {result.failed_events}")
print(f"Duration      : {result.duration_seconds:.2f}s")
if result.errors:
    print(f"Errors        : {result.errors}")

2026-02-28 21:01:50,463 [INFO] pipeline.ticketmaster: Starting multi-city execution: ticketmaster_20260228_200150_39493678 (2 cities)


2026-02-28 21:01:50,464 [INFO] pipeline.ticketmaster: Fetching events for Barcelona...


2026-02-28 21:01:50,464 [INFO] pipeline.ticketmaster:   Barcelona: sliding window fetch [2026-02-28..2026-06-28] (capacity=1000/call, window=168h)


2026-02-28 21:01:50,465 [INFO] src.ingestion.pipelines.apis.base_api: Fetching page 0/4...


2026-02-28 21:01:51,111 [INFO] httpx: HTTP Request: GET https://app.ticketmaster.com/discovery/v2/events.json?apikey=guGHnH0k1CTQfmSGl9vhBDU1JLV3GW0w&city=Barcelona&countryCode=ES&startDateTime=2026-02-28T00%3A00%3A00Z&endDateTime=2026-03-06T23%3A59%3A59Z&size=200&page=0&sort=date%2Casc "HTTP/1.1 200 OK"


...

2026-02-28 21:02:08,689 [INFO] pipeline.ticketmaster:   Madrid: 1433 raw events fetched


2026-02-28 21:02:08,689 [INFO] pipeline.ticketmaster: Total raw events across all cities: 1481


2026-02-28 21:02:13,982 [INFO] pipeline.ticketmaster: Deduplication: 1481 -> 1481 events


2026-02-28 21:02:13,982 [INFO] pipeline.ticketmaster: Multi-city pipeline completed: 1481/1481 successful


Status        : PipelineStatus.SUCCESS
Execution ID  : ticketmaster_20260228_200150_39493678
Total fetched : 1481
Successful    : 1481
Failed        : 0
Duration      : 23.52s


In [4]:
print(f"Inspecting {len(result.events)} normalised events\n")
print("-" * 70)

for i, event in enumerate(result.events):
    venue = event.location.venue_name or "N/A"
    city  = event.location.city or "N/A"
    start = event.start_datetime.isoformat() if event.start_datetime else "N/A"
    etype = (event.event_type.value if hasattr(event.event_type, "value") else str(event.event_type)) if event.event_type else "N/A"
    cf_keys = list(event.custom_fields.keys()) if event.custom_fields else []

    print(f"[{i+1:02d}] {event.title}")
    print(f"     Venue  : {venue} ({city})")
    print(f"     Date   : {start}")
    print(f"     Type   : {etype}")
    print(f"     CF keys: {cf_keys}")
    print()

Inspecting 1481 normalised events

----------------------------------------------------------------------
[01] Vida Records & Friends: Presentació Cartell + Mujeres
     Venue  : Antiga Fàbrica Estrella Damm (Barcelona)
     Date   : 2026-03-03T18:00:00+00:00
     Type   : concert
     CF keys: ['event_status', 'sale_start', 'sale_end', 'promoter_name', 'promoter_id', 'classification_segment', 'classification_genre', 'classification_subgenre', 'country_name', 'attraction_genre']

[02] The Kooks
     Venue  : Sala Razzmatazz 1 (Barcelona)
     Date   : 2026-03-03T19:30:00+00:00
     Type   : concert
     CF keys: ['event_status', 'sale_start', 'sale_end', 'promoter_name', 'promoter_id', 'classification_segment', 'classification_genre', 'classification_subgenre', 'country_name', 'attraction_genre']

[03] MEDUZA - MEDUZA3
     Venue  : Sala Razzmatazz 1 (Barcelona)
     Date   : 2026-03-04T19:30:00+00:00
     Type   : concert
     CF keys: ['event_status', 'sale_start', 'sale_end', 'promo

In [5]:
print("custom_fields inspection for Ticketmaster\n")
print("-" * 70)

for i, event in enumerate(result.events[:3]):
    print(f"Event: {event.title}")
    if event.custom_fields:
        for key, val in event.custom_fields.items():
            print(f"  {key}: {val}")
    else:
        print("  (no custom_fields)")
    print()

# Aggregate all custom_field keys across all events
all_cf_keys = set()
for event in result.events:
    if event.custom_fields:
        all_cf_keys.update(event.custom_fields.keys())
print(f"All custom_fields keys found: {sorted(all_cf_keys)}")

custom_fields inspection for Ticketmaster

----------------------------------------------------------------------
Event: Vida Records & Friends: Presentació Cartell + Mujeres
  event_status: onsale
  sale_start: 2025-07-29T06:00:00Z
  sale_end: 2026-03-03T19:00:00Z
  promoter_name: Sitback Produccions, S.L.
  promoter_id: 4021
  classification_segment: Music
  classification_genre: Alternative
  classification_subgenre: Alternative Rock
  country_name: Spain
  attraction_genre: Alternative

Event: The Kooks
  event_status: onsale
  sale_start: 2025-09-19T08:00:00Z
  sale_end: 2026-03-03T19:30:00Z
  promoter_name: Live Nation España S.A.U.
  promoter_id: 2727
  classification_segment: Music
  classification_genre: Alternative
  classification_subgenre: Alternative Rock
  country_name: Spain
  attraction_genre: Rock

Event: MEDUZA - MEDUZA3
  event_status: onsale
  sale_start: 2025-10-17T08:30:00Z
  sale_end: 2026-03-04T19:30:00Z
  promoter_name: Live Nation España S.A.U.
  promoter_id: 

In [6]:
from src.ingestion.batch_writer import BatchWriter

writer = BatchWriter()

batch_path = writer.save_batch(
    source_name="ticketmaster",
    events=result.events,
    batch_id=result.execution_id,
    metadata={
        "pipeline_status": result.status.value,
        "duration_seconds": result.duration_seconds,
    },
)

print(f"Batch saved to : {batch_path}")
print(f"File size      : {batch_path.stat().st_size:,} bytes")

2026-02-28 21:02:14,104 [INFO] src.ingestion.batch_writer: BatchWriter: saved 1481/1481 events for 'ticketmaster' → /Users/josegarcia/Documents/GitHub/event-intelligence-platform/data/batches/ticketmaster/2026-02-28_ticketmaster_20260228_200150_39493678.jsonl


Batch saved to : /Users/josegarcia/Documents/GitHub/event-intelligence-platform/data/batches/ticketmaster/2026-02-28_ticketmaster_20260228_200150_39493678.jsonl
File size      : 10,170,197 bytes


In [7]:
import pandas as pd

if result.events:
    df = pipeline.to_dataframe(result.events)
    print(f"DataFrame shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print("\nBasic stats:")
    print(f"  Events          : {len(df)}")
    print(f"  Unique cities   : {df['city'].nunique()}")
    print(f"  Avg quality     : {df['data_quality_score'].mean():.2f}")
    print(f"  Free events     : {df['price_is_free'].sum()}")
    print("\nSample (title, city, start_datetime, data_quality_score):")
    print(df[["title", "city", "start_datetime", "data_quality_score"]].to_string(index=False))
else:
    print("No events to display.")

DataFrame shape: (1481, 79)

Columns: ['event_id', 'title', 'description', 'start_datetime', 'end_datetime', 'duration_minutes', 'is_all_day', 'is_recurring', 'recurrence_pattern', 'venue_name', 'street_address', 'city', 'state_or_region', 'postal_code', 'country_code', 'latitude', 'longitude', 'timezone', 'event_type', 'event_format', 'capacity', 'age_restriction', 'price_currency', 'price_is_free', 'price_minimum', 'price_maximum', 'price_early_bird', 'price_standard', 'price_vip', 'price_raw_text', 'ticket_url', 'ticket_is_sold_out', 'ticket_count_available', 'ticket_early_bird_deadline', 'organizer_name', 'organizer_url', 'organizer_email', 'organizer_phone', 'organizer_image_url', 'organizer_follower_count', 'organizer_verified', 'source_name', 'source_event_id', 'source_url', 'source_updated_at', 'source_ingestion_timestamp', 'media_assets_json', 'engagement_going_count', 'engagement_interested_count', 'engagement_views_count', 'engagement_shares_count', 'engagement_comments_coun

In [8]:
await pipeline.close()
print("Pipeline closed. Resources released.")

Pipeline closed. Resources released.
